# Rule-based text preparation — VietMed

Notebook này **chỉ tải cột `text`** từ dataset `leduckhai/VietMed` và xử lý riêng phần rule-based normalization.

Không tải/giải mã audio, không chạy ASR, không chạy NER. Mục tiêu là kiểm tra corpus text và các luật chuẩn hóa trước khi xây medical lexicon correction.

Các split được xử lý: `train`, `dev`, `test`, `cv` nếu split tồn tại.

In [ ]:
!pip -q install -U datasets pandas

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from datasets import DatasetNotFoundError, load_dataset

DATASET_ID = 'leduckhai/VietMed'
REQUESTED_SPLITS = ['train', 'dev', 'test', 'cv']
OUTPUT_DIR = Path('/content/vietmed_text_rulebase')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thử trực tiếp các split đã biết để tránh gọi API metadata có thể bị giới hạn.
# Nếu Hub vẫn giới hạn request, chạy riêng: from huggingface_hub import login; login()
splits = REQUESTED_SPLITS
print('Sẽ thử trực tiếp các split:', splits)

In [ ]:
# Tải duy nhất cột text và metadata nhẹ; không lấy/giải mã cột audio
text_records = {}
requested_columns = [
    'text', 'duration', 'utterance_id', 'audio_name', 'role',
    'gender', 'accent', 'icd10_code', 'rec_condition'
]
for split in splits:
    try:
        dataset = load_dataset(DATASET_ID, split=split, streaming=True)
        available_columns = list(dataset.features)
        columns = [column for column in requested_columns if column in available_columns]
        if 'text' not in columns:
            print(split, 'bỏ qua: không có cột text')
            continue
        dataset = dataset.select_columns(columns)
        records = []
        for index, row in enumerate(dataset):
            records.append({'split': split, 'index': index, **row})
        text_records[split] = records
        print(split, 'records:', len(records), 'columns:', columns)
    except DatasetNotFoundError:
        print(split, 'bỏ qua: split không tồn tại')

all_records = [record for records in text_records.values() for record in records]
print('Total records:', len(all_records))

In [ ]:
# Lưu text raw — đây là corpus chuẩn, chưa phải ASR noisy transcript
raw_path = OUTPUT_DIR / 'vietmed_text_raw.jsonl'
raw_path.write_text(
    ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in all_records),
    encoding='utf-8',
)
raw_df = pd.DataFrame(all_records)
raw_df.to_csv(OUTPUT_DIR / 'vietmed_text_raw.csv', index=False, encoding='utf-8-sig')
print('Saved:', raw_path, OUTPUT_DIR / 'vietmed_text_raw.csv')

In [ ]:
# Các luật normalization hình thức, không thay đổi từ theo ngữ nghĩa
CONTROL_CHARS = re.compile(r'[\x00-\x1f\x7f-\x9f]')
MULTIPLE_SPACES = re.compile(r'\s+')
SPACE_BEFORE_PUNCT = re.compile(r'\s+([,.;:!?%])')
SPACE_AFTER_PUNCT = re.compile(r'([,.;:!?])(?=\S)')

def normalize_text(text):
    if not isinstance(text, str):
        return '', ['non_string_or_null']
    value = unicodedata.normalize('NFC', text)
    value = CONTROL_CHARS.sub(' ', value)
    value = SPACE_BEFORE_PUNCT.sub(r'\1', value)
    value = SPACE_AFTER_PUNCT.sub(r'\1 ', value)
    value = MULTIPLE_SPACES.sub(' ', value).strip()
    return value, [] if value == text else ['formatting_normalization']

normalized_records = []
for record in all_records:
    normalized, rules = normalize_text(record.get('text'))
    normalized_records.append({
        **record,
        'normalized_text': normalized,
        'changed': normalized != record.get('text', ''),
        'rules_applied': rules,
    })

normalized_path = OUTPUT_DIR / 'vietmed_text_normalized.jsonl'
normalized_path.write_text(
    ''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in normalized_records),
    encoding='utf-8',
)
pd.DataFrame(normalized_records).to_csv(OUTPUT_DIR / 'vietmed_text_normalized.csv', index=False, encoding='utf-8-sig')
print('Changed:', sum(record['changed'] for record in normalized_records), '/', len(normalized_records))
print('Saved:', normalized_path)

In [ ]:
# Thống kê corpus và thay đổi normalization
summary_rows = []
for split, records in text_records.items():
    subset = [record for record in normalized_records if record['split'] == split]
    summary_rows.append({
        'split': split,
        'records': len(subset),
        'empty_text': sum(not isinstance(record.get('text'), str) or not record.get('text', '').strip() for record in subset),
        'changed_by_normalization': sum(record['changed'] for record in subset),
        'avg_chars_raw': round(sum(len(record.get('text', '') or '') for record in subset) / len(subset), 2) if subset else 0,
        'avg_chars_normalized': round(sum(len(record['normalized_text']) for record in subset) / len(subset), 2) if subset else 0,
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'text_rulebase_summary.csv', index=False, encoding='utf-8-sig')
summary = {
    'dataset': DATASET_ID,
    'splits': splits,
    'total_records': len(normalized_records),
    'changed_by_normalization': sum(record['changed'] for record in normalized_records),
    'empty_text': sum(not record['normalized_text'] for record in normalized_records),
    'warning': 'These are clean/reference texts only. They are not ASR noisy transcripts and cannot by themselves measure ASR correction accuracy.',
}
(OUTPUT_DIR / 'text_rulebase_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(summary_df.to_string(index=False))
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# Tải các file kết quả về máy
import shutil
archive = shutil.make_archive('/content/vietmed_text_rulebase', 'zip', OUTPUT_DIR)
print('ZIP:', archive)
from google.colab import files
files.download(archive)

## Kết luận phạm vi

- File `vietmed_text_raw.*` là transcript chuẩn lấy từ cột `text`.
- File `vietmed_text_normalized.*` là transcript sau normalization hình thức.
- Nếu số câu thay đổi bằng 0, điều đó chỉ có nghĩa text đã sạch về mặt format.
- Không thể học lỗi ASR thực tế từ `text` chuẩn một mình. Muốn sửa lỗi như `chày máu não → chảy máu não`, cần ASR transcript noisy tương ứng hoặc tạo lỗi giả có quy tắc rõ ràng.
- Chỉ dùng text train/dev để xây lexicon; không dùng text test để tạo rule.